## **Imports**

In [36]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

RANDOM_STATE = 42

TRAIN_FILE = "/content/drive/MyDrive/dataset_C_training.csv"
TEST_FILE = "/content/drive/MyDrive/dataset_C_testing.csv"
EXAMPLE_FILE = "/content/drive/MyDrive/dataset_C_example_submission.csv"

TARGET_COL = "covid_vaccine"
ID_COL = "respondent_id"

OUTPUT_FOLDER = Path("outputs")
PLOT_FOLDER = OUTPUT_FOLDER / "plotly_plots"
SUBMISSION_FOLDER = OUTPUT_FOLDER / "submissions"

OUTPUT_FOLDER.mkdir(exist_ok=True)
PLOT_FOLDER.mkdir(exist_ok=True)
SUBMISSION_FOLDER.mkdir(exist_ok=True)

team_members = {"person01": "Mahesh", "person02": "Waleed", "person03": "Gopal", "person04": "Lavanya", "person05": "Rakesh", "person06": "Thivas"}
print("Setup complete.")

Setup complete.


## **Load files**

In [37]:
train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)
example_submission = pd.read_csv(EXAMPLE_FILE)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Example submission shape:", example_submission.shape)

print("\nTrain head:")
display(train_df.head())

print("\nTest head:")
display(test_df.head())

print("\nExample submission head:")
display(example_submission.head())

print("\nExample submission columns:")
print(example_submission.columns.tolist())

Train shape: (4756, 31)
Test shape: (4749, 30)
Example submission shape: (4749, 2)

Train head:


,respondent_id,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,employment_status,census_msa,household_adults,household_children,doctor_recc_covid,opinion_covid_vacc_effective,opinion_covid_risk,opinion_covid_sick_from_vacc,employment_sector,covid_vaccine
0,1,3.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Employed,"MSA, Principle City",3.0,2.0,0,4,4,2.0,construction,0
1,2,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Employed,Non-MSA,0.0,0.0,0,5,2,1.0,education,1
2,3,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Employed,"MSA, Not Principle City",0.0,0.0,0,2,2,5.0,wholesale,0
3,4,2.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Not in Labor Force,"MSA, Not Principle City",1.0,0.0,1,3,3,2.0,NaN,1
4,5,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,Employed,"MSA, Not Principle City",0.0,0.0,0,3,2,2.0,wholesale,0



Test head:


,respondent_id,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,rent_or_own,employment_status,census_msa,household_adults,household_children,doctor_recc_covid,opinion_covid_vacc_effective,opinion_covid_risk,opinion_covid_sick_from_vacc,employment_sector
0,4757,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,Own,Not in Labor Force,"MSA, Principle City",0.0,0.0,0,4,2,2.0,NaN
1,4758,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,NaN,NaN,"MSA, Not Principle City",0.0,0.0,0,2,3,2.0,NaN
2,4759,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,Own,Not in Labor Force,"MSA, Principle City",1.0,0.0,0,3,2,2.0,NaN
3,4760,3.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,Rent,Employed,Non-MSA,1.0,0.0,0,4,4,1.0,agriculture
4,4761,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Own,Employed,Non-MSA,0.0,0.0,0,3,2,1.0,wholesale



Example submission head:


,respondent_id,covid_vaccine
0,4757,0.0
1,4758,0.0
2,4759,0.0
3,4760,0.0
4,4761,0.0



Example submission columns:
['respondent_id', 'covid_vaccine']


## **Cleaning hidden missing files**

In [38]:
fake_missing_values = [
    "", " ", "NA", "N/A", "na", "n/a",
    "NULL", "null", "None", "none",
    "?", "-", "--"
]

train_df = train_df.replace(fake_missing_values, np.nan)
test_df = test_df.replace(fake_missing_values, np.nan)

print("Hidden missing values cleaned.")

Hidden missing values cleaned.


## **Basic data overview**

In [39]:
overview_df = pd.DataFrame()
overview_df["column"] = train_df.columns
overview_df["data_type"] = train_df.dtypes.astype(str).values
overview_df["missing_count"] = train_df.isna().sum().values
overview_df["missing_rate"] = train_df.isna().mean().values
overview_df["unique_values"] = train_df.nunique(dropna=True).values

display(overview_df)
overview_df.to_csv(OUTPUT_FOLDER / "dataset_overview.csv", index=False)

print("Target value counts:")
display(train_df[TARGET_COL].value_counts())
print("Target proportions:")
display(train_df[TARGET_COL].value_counts(normalize=True))

,column,data_type,missing_count,missing_rate,unique_values
0,respondent_id,int64,0,0.000000,4756
1,covid_concern,float64,12,0.002523,4
2,covid_knowledge,float64,27,0.005677,3
3,behavioral_antiviral_meds,float64,14,0.002944,2
4,behavioral_avoidance,float64,27,0.005677,2
5,behavioral_face_mask,float64,2,0.000421,2
6,behavioral_wash_hands,float64,9,0.001892,2
7,behavioral_large_gatherings,float64,10,0.002103,2
8,behavioral_outside_home,float64,14,0.002944,2
9,behavioral_touch_face,float64,21,0.004415,2


Target value counts:


,count
covid_vaccine,
0,3202
1,1554


Target proportions:


,proportion
covid_vaccine,
0,0.673255
1,0.326745


## **Plot target distribution**

In [40]:
target_plot_df = train_df[TARGET_COL].value_counts().sort_index().reset_index()
target_plot_df.columns = [TARGET_COL, "count"]

fig = px.bar(target_plot_df, x=TARGET_COL, y="count", text="count", title="Target Distribution")
fig.update_layout(xaxis_title="covid_vaccine", yaxis_title="Count")
fig.show()
fig.write_html(PLOT_FOLDER / "target_distribution.html")

## **Missing value table and plot**

In [41]:
train_missing = train_df.isna().mean()
test_missing = test_df.isna().mean()
missing_df = pd.DataFrame()
missing_df["column"] = train_missing.index
missing_df["train_missing_rate"] = train_missing.values
missing_df["test_missing_rate"] = missing_df["column"].map(test_missing).fillna(0).values
missing_df = missing_df.sort_values("train_missing_rate", ascending=False)
display(missing_df[missing_df["train_missing_rate"] > 0])
missing_df.to_csv(OUTPUT_FOLDER / "missing_value_summary.csv", index=False)
missing_plot_df = missing_df[(missing_df["train_missing_rate"] > 0) | (missing_df["test_missing_rate"] > 0)]
fig = go.Figure()

fig.add_trace(go.Bar(y=missing_plot_df["column"], x=missing_plot_df["train_missing_rate"], name="Train", orientation="h"))
fig.add_trace(go.Bar(y=missing_plot_df["column"], x=missing_plot_df["test_missing_rate"], name="Test", orientation="h"))
fig.update_layout(title="Missing Value Rate: Train vs Test", xaxis_title="Missing rate", yaxis_title="Column", barmode="group",
    height=max(500, 28 * len(missing_plot_df))
)

fig.show()
fig.write_html(PLOT_FOLDER / "missing_value_rates.html")

,column,train_missing_rate,test_missing_rate
29,employment_sector,0.489907,0.506422
13,health_insurance,0.401177,0.417351
18,income_poverty,0.157275,0.167825
20,rent_or_own,0.069176,0.079806
21,employment_status,0.050042,0.058328
19,marital_status,0.048570,0.056644
15,education,0.047939,0.056644
10,chronic_med_condition,0.035534,0.035376
11,child_under_6_months,0.028385,0.032638
12,health_worker,0.027754,0.031375


## **Missingness heatmap plot**

In [42]:
missing_matrix = train_df.isna().astype(int)
if len(missing_matrix) > 1000:
    missing_matrix_small = missing_matrix.sample(1000, random_state=RANDOM_STATE)
else:
    missing_matrix_small = missing_matrix.copy()

fig = px.imshow(missing_matrix_small.T, aspect="auto", title="Missingness Heatmap")
fig.update_layout(xaxis_title="Sampled row number", yaxis_title="Column")
fig.show()
fig.write_html(PLOT_FOLDER / "missingness_heatmap.html")

## **Put columns into groups**

In [43]:
binary_cols = [
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face",
    "doctor_recc_covid",
    "chronic_med_condition",
    "child_under_6_months",
    "health_worker",
    "health_insurance"
]

ordinal_number_cols = [
    "covid_concern",
    "covid_knowledge",
    "opinion_covid_vacc_effective",
    "opinion_covid_risk",
    "opinion_covid_sick_from_vacc",
    "household_adults",
    "household_children"
]

ordinal_text_cols = [
    "age_group",
    "education",
    "income_poverty"
]

normal_category_cols = [
    "race",
    "sex",
    "marital_status",
    "rent_or_own",
    "employment_status",
    "census_msa",
    "employment_sector"
]

binary_cols = [col for col in binary_cols if col in train_df.columns]
ordinal_number_cols = [col for col in ordinal_number_cols if col in train_df.columns]
ordinal_text_cols = [col for col in ordinal_text_cols if col in train_df.columns]
normal_category_cols = [col for col in normal_category_cols if col in train_df.columns]

print("Binary columns:", binary_cols)
print("Ordinal number columns:", ordinal_number_cols)
print("Ordinal text columns:", ordinal_text_cols)
print("Normal category columns:", normal_category_cols)

Binary columns: ['behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_covid', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance']
Ordinal number columns: ['covid_concern', 'covid_knowledge', 'opinion_covid_vacc_effective', 'opinion_covid_risk', 'opinion_covid_sick_from_vacc', 'household_adults', 'household_children']
Ordinal text columns: ['age_group', 'education', 'income_poverty']
Normal category columns: ['race', 'sex', 'marital_status', 'rent_or_own', 'employment_status', 'census_msa', 'employment_sector']


## **Binary feature plots against targets**

In [44]:
binary_target_list = []

for col in binary_cols:
    temp_df = train_df.copy()
    temp_df[col] = temp_df[col].fillna("Missing")
    grouped = temp_df.groupby(col)[TARGET_COL].mean().reset_index()
    grouped["feature"] = col
    grouped.columns = ["value", "vaccine_rate", "feature"]
    binary_target_list.append(grouped)

binary_target_df = pd.concat(binary_target_list, ignore_index=True)

fig = px.bar(binary_target_df, x="feature", y="vaccine_rate", color="value", barmode="group", title="Vaccine Rate by Binary Features")
fig.update_layout(xaxis_title="Feature", yaxis_title="Mean covid_vaccine", xaxis_tickangle=-45)
fig.show()
fig.write_html(PLOT_FOLDER / "binary_features_vs_target.html")

## **Ordinal feature target plots**

In [45]:
for col in ordinal_number_cols:
    temp_df = train_df.copy()
    temp_df[col] = temp_df[col].fillna("Missing")
    plot_df = temp_df.groupby(col)[TARGET_COL].mean().reset_index()
    plot_df.columns = [col, "vaccine_rate"]

    fig = px.bar(plot_df,x=col, y="vaccine_rate", text="vaccine_rate", title=f"Vaccine Rate by {col}")
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    fig.update_layout(yaxis_title="Mean covid_vaccine")
    fig.show()
    fig.write_html(PLOT_FOLDER / f"target_rate_by_{col}.html")

## **Categorical feature target plots**

In [46]:
category_plot_cols = ordinal_text_cols + normal_category_cols

for col in category_plot_cols:
    temp_df = train_df.copy()
    temp_df[col] = temp_df[col].fillna("Missing")
    counts = temp_df[col].value_counts()
    keep_values = counts[counts >= 20].index
    temp_df = temp_df[temp_df[col].isin(keep_values)]
    plot_df = temp_df.groupby(col)[TARGET_COL].mean().reset_index()
    plot_df = plot_df.sort_values(TARGET_COL, ascending=False)

    fig = px.bar(plot_df, x=col, y=TARGET_COL, text=TARGET_COL, title=f"Vaccine Rate by {col}")
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    fig.update_layout(xaxis_title=col, yaxis_title="Mean covid_vaccine", xaxis_tickangle=-45)
    fig.show()
    fig.write_html(PLOT_FOLDER / f"target_rate_by_{col}.html")

## **Correlation heatmap plot**

In [47]:
corr_cols = binary_cols + ordinal_number_cols + [TARGET_COL]
corr_cols = [col for col in corr_cols if col in train_df.columns]
corr_df = train_df[corr_cols].corr(numeric_only=True)

fig = px.imshow(corr_df, text_auto=".2f", aspect="auto", title="Correlation Heatmap")
fig.show()
fig.write_html(PLOT_FOLDER / "correlation_heatmap.html")
target_corr = corr_df[TARGET_COL].drop(TARGET_COL).sort_values(ascending=False)
target_corr_df = target_corr.reset_index()
target_corr_df.columns = ["feature", "correlation_with_target"]

fig = px.bar(target_corr_df, x="feature", y="correlation_with_target", text="correlation_with_target", title="Correlation with Target")
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(xaxis_tickangle=-45)
fig.show()
fig.write_html(PLOT_FOLDER / "target_correlation_barplot.html")

## **Training vs test missing plot**

In [48]:
fig = px.scatter(missing_df, x="train_missing_rate", y="test_missing_rate", text="column", title="Train vs Test Missing Rate")
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="same missing rate"))
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Train missing rate", yaxis_title="Test missing rate")
fig.show()
fig.write_html(PLOT_FOLDER / "train_vs_test_missing_rate.html")

## **Train vs test shift check**

In [50]:
shift_rows = []
check_cols = binary_cols + ordinal_number_cols + ordinal_text_cols + normal_category_cols
for col in check_cols:
    if col in train_df.columns and col in test_df.columns:
        train_rates = train_df[col].fillna("Missing").astype(str).value_counts(normalize=True)
        test_rates = test_df[col].fillna("Missing").astype(str).value_counts(normalize=True)
        all_values = train_rates.index.union(test_rates.index)
        biggest_difference = 0
        biggest_value = None

        for value in all_values:
            train_value_rate = train_rates.get(value, 0)
            test_value_rate = test_rates.get(value, 0)
            difference = abs(train_value_rate - test_value_rate)

            if difference > biggest_difference:
                biggest_difference = difference
                biggest_value = value

        shift_rows.append({
            "column": col,
            "most_shifted_value": biggest_value,
            "biggest_train_test_difference": biggest_difference
        })

shift_df = pd.DataFrame(shift_rows)
shift_df = shift_df.sort_values("biggest_train_test_difference", ascending=False)
display(shift_df)
shift_df.to_csv(OUTPUT_FOLDER / "train_test_shift_summary.csv", index=False)

fig = px.bar(shift_df.head(15), x="column", y="biggest_train_test_difference", text="biggest_train_test_difference", title="Top Train vs Test Distribution Differences")
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(xaxis_tickangle=-45, yaxis_title="Largest category proportion difference")
fig.show()
fig.write_html(PLOT_FOLDER / "train_test_shift_summary.html")

,column,most_shifted_value,biggest_train_test_difference
12,covid_concern,3.0,0.021317
19,age_group,18 - 34 Years,0.018658
14,opinion_covid_vacc_effective,4,0.018200
21,income_poverty,"> $75,000",0.017497
28,employment_sector,Missing,0.016515
11,health_insurance,Missing,0.016174
26,employment_status,Employed,0.015454
27,census_msa,Non-MSA,0.014913
15,opinion_covid_risk,4,0.014006
5,behavioral_outside_home,1.0,0.013960


## **Train and validation split**

In [52]:
X = train_df.drop(columns=[TARGET_COL])
y = train_df[TARGET_COL].astype(int)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
X_test_raw = test_df.copy()

print("X_train_raw:", X_train_raw.shape)
print("X_val_raw:", X_val_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("\nTrain target balance:")
display(y_train.value_counts(normalize=True))
print("\nValidation target balance:")
display(y_val.value_counts(normalize=True))

X_train_raw: (3804, 30)
X_val_raw: (952, 30)
X_test_raw: (4749, 30)

Train target balance:


,proportion
covid_vaccine,
0,0.673239
1,0.326761



Validation target balance:


,proportion
covid_vaccine,
0,0.673319
1,0.326681


## **Remove ID from model features**

In [53]:
train_ids = X_train_raw[ID_COL].copy()
val_ids = X_val_raw[ID_COL].copy()
test_ids = X_test_raw[ID_COL].copy()
X_train_raw = X_train_raw.drop(columns=[ID_COL])
X_val_raw = X_val_raw.drop(columns=[ID_COL])
X_test_raw = X_test_raw.drop(columns=[ID_COL])

print("ID column removed from features.")
print("X_train_raw:", X_train_raw.shape)
print("X_val_raw:", X_val_raw.shape)
print("X_test_raw:", X_test_raw.shape)

ID column removed from features.
X_train_raw: (3804, 29)
X_val_raw: (952, 29)
X_test_raw: (4749, 29)


## **Add simple engineered features**

In [54]:
behavior_cols = [
    "behavioral_antiviral_meds",
    "behavioral_avoidance",
    "behavioral_face_mask",
    "behavioral_wash_hands",
    "behavioral_large_gatherings",
    "behavioral_outside_home",
    "behavioral_touch_face"
]

opinion_cols = [
    "opinion_covid_vacc_effective",
    "opinion_covid_risk",
    "opinion_covid_sick_from_vacc"
]

high_missing_cols = [
    "employment_sector",
    "health_insurance",
    "income_poverty"
]

def add_features(df):
    df = df.copy()

    for col in high_missing_cols:
        if col in df.columns:
            df[col + "_was_missing"] = df[col].isna().astype(int)

    behavior_cols_that_exist = [col for col in behavior_cols if col in df.columns]
    df["behavior_score"] = df[behavior_cols_that_exist].sum(axis=1, skipna=True)
    df["behavior_missing_count"] = df[behavior_cols_that_exist].isna().sum(axis=1)
    df["behavior_observed_count"] = df[behavior_cols_that_exist].notna().sum(axis=1)

    opinion_cols_that_exist = [col for col in opinion_cols if col in df.columns]
    df["opinion_missing_count"] = df[opinion_cols_that_exist].isna().sum(axis=1)
    df["opinion_observed_count"] = df[opinion_cols_that_exist].notna().sum(axis=1)

    df["covid_opinion_positive_score"] = (
        df["opinion_covid_vacc_effective"].fillna(0)
        + df["opinion_covid_risk"].fillna(0)
        - df["opinion_covid_sick_from_vacc"].fillna(0)
    )

    df["household_total"] = (
        df["household_adults"].fillna(0)
        + df["household_children"].fillna(0)
    )

    df["household_missing_count"] = (
        df[["household_adults", "household_children"]].isna().sum(axis=1)
    )

    df["doctor_recc_x_risk"] = (
        df["doctor_recc_covid"].fillna(0)
        * df["opinion_covid_risk"].fillna(0)
    )

    df["behavior_score_bin"] = pd.cut(
        df["behavior_score"],
        bins=[-0.1, 1, 3, 7],
        labels=["low_behavior", "medium_behavior", "high_behavior"]
    ).astype("object")

    df["household_total_bin"] = pd.cut(
        df["household_total"],
        bins=[-0.1, 0, 2, 10],
        labels=["alone", "small_household", "large_household"]
    ).astype("object")

    return df

X_train_fe = add_features(X_train_raw)
X_val_fe = add_features(X_val_raw)
X_test_fe = add_features(X_test_raw)

print("Feature engineering done.")
print("Train:", X_train_fe.shape)
print("Validation:", X_val_fe.shape)
print("Test:", X_test_fe.shape)

Feature engineering done.
Train: (3804, 43)
Validation: (952, 43)
Test: (4749, 43)


## **Make ordinal encoded features**

In [55]:
def add_ordinal_features(df):
    df = df.copy()

    if "covid_concern" in df.columns:
        df["covid_concern_ord"] = df["covid_concern"].map({0: 1, 1: 2, 2: 3, 3: 4}).fillna(0).astype(int)

    if "covid_knowledge" in df.columns:
        df["covid_knowledge_ord"] = df["covid_knowledge"].map({0: 1, 1: 2, 2: 3}).fillna(0).astype(int)

    opinion_order_cols = [
        "opinion_covid_vacc_effective",
        "opinion_covid_risk",
        "opinion_covid_sick_from_vacc"
    ]

    for col in opinion_order_cols:
        if col in df.columns:
            df[col + "_ord"] = df[col].map({1: 1, 2: 2, 3: 3, 4: 4, 5: 5}).fillna(0).astype(int)

    for col in ["household_adults", "household_children"]:
        if col in df.columns:
            df[col + "_ord"] = df[col].map({0: 1, 1: 2, 2: 3, 3: 4}).fillna(0).astype(int)

    if "age_group" in df.columns:
        df["age_group_ord"] = df["age_group"].map({
            "18 - 34 Years": 1,
            "35 - 44 Years": 2,
            "45 - 54 Years": 3,
            "55 - 64 Years": 4,
            "65+ Years": 5
        }).fillna(0).astype(int)

    if "education" in df.columns:
        df["education_ord"] = df["education"].map({
            "< 12 Years": 1,
            "12 Years": 2,
            "Some College": 3,
            "College Graduate": 4
        }).fillna(0).astype(int)

    if "income_poverty" in df.columns:
        df["income_poverty_ord"] = df["income_poverty"].map({
            "Below Poverty": 1,
            "<= $75,000, Above Poverty": 2,
            "> $75,000": 3
        }).fillna(0).astype(int)

    return df

X_train_ready = add_ordinal_features(X_train_fe)
X_val_ready = add_ordinal_features(X_val_fe)
X_test_ready = add_ordinal_features(X_test_fe)

print("Ordinal features added.")

Ordinal features added.


## **Pick columns for final encoding**

In [56]:
binned_cols = [
    "behavior_score_bin",
    "household_total_bin"
]

onehot_cols = binary_cols + normal_category_cols + binned_cols
onehot_cols = [col for col in onehot_cols if col in X_train_ready.columns]

ordinal_cols = [
    "covid_concern_ord",
    "covid_knowledge_ord",
    "opinion_covid_vacc_effective_ord",
    "opinion_covid_risk_ord",
    "opinion_covid_sick_from_vacc_ord",
    "household_adults_ord",
    "household_children_ord",
    "age_group_ord",
    "education_ord",
    "income_poverty_ord"
]

ordinal_cols = [col for col in ordinal_cols if col in X_train_ready.columns]

extra_number_cols = [
    "employment_sector_was_missing",
    "health_insurance_was_missing",
    "income_poverty_was_missing",
    "behavior_score",
    "behavior_missing_count",
    "behavior_observed_count",
    "opinion_missing_count",
    "opinion_observed_count",
    "covid_opinion_positive_score",
    "household_total",
    "household_missing_count",
    "doctor_recc_x_risk"
]

extra_number_cols = [col for col in extra_number_cols if col in X_train_ready.columns]

number_cols = ordinal_cols + extra_number_cols

print("One-hot columns:")
print(onehot_cols)
print("\nNumber columns:")
print(number_cols)

One-hot columns:
['behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_covid', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance', 'race', 'sex', 'marital_status', 'rent_or_own', 'employment_status', 'census_msa', 'employment_sector', 'behavior_score_bin', 'household_total_bin']

Number columns:
['covid_concern_ord', 'covid_knowledge_ord', 'opinion_covid_vacc_effective_ord', 'opinion_covid_risk_ord', 'opinion_covid_sick_from_vacc_ord', 'household_adults_ord', 'household_children_ord', 'age_group_ord', 'education_ord', 'income_poverty_ord', 'employment_sector_was_missing', 'health_insurance_was_missing', 'income_poverty_was_missing', 'behavior_score', 'behavior_missing_count', 'behavior_observed_count', 'opinion_missing_count', 'opinion_observed_count', 'covid_opinion_positive_score', 'household_total', 'househ

## **One hot encode missing as a category**

In [57]:
X_train_onehot_part = X_train_ready[onehot_cols].copy()
X_val_onehot_part = X_val_ready[onehot_cols].copy()
X_test_onehot_part = X_test_ready[onehot_cols].copy()
X_train_onehot_part = X_train_onehot_part.fillna("Missing").astype(str)
X_val_onehot_part = X_val_onehot_part.fillna("Missing").astype(str)
X_test_onehot_part = X_test_onehot_part.fillna("Missing").astype(str)

try:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

X_train_onehot = onehot_encoder.fit_transform(X_train_onehot_part)
X_val_onehot = onehot_encoder.transform(X_val_onehot_part)
X_test_onehot = onehot_encoder.transform(X_test_onehot_part)
onehot_feature_names = onehot_encoder.get_feature_names_out(onehot_cols)

print("One-hot encoding done.")
print("One-hot train shape:", X_train_onehot.shape)

One-hot encoding done.
One-hot train shape: (3804, 82)


## **Joining one hot and numeric final features**

In [58]:
X_train_number_part = X_train_ready[number_cols].copy()
X_val_number_part = X_val_ready[number_cols].copy()
X_test_number_part = X_test_ready[number_cols].copy()
X_train_number_part = X_train_number_part.fillna(0)
X_val_number_part = X_val_number_part.fillna(0)
X_test_number_part = X_test_number_part.fillna(0)
X_train_onehot_df = pd.DataFrame(X_train_onehot, columns=onehot_feature_names, index=X_train_ready.index)
X_val_onehot_df = pd.DataFrame(X_val_onehot, columns=onehot_feature_names, index=X_val_ready.index)
X_test_onehot_df = pd.DataFrame(X_test_onehot, columns=onehot_feature_names, index=X_test_ready.index)
X_train_model = pd.concat([X_train_onehot_df, X_train_number_part], axis=1)
X_val_model = pd.concat([X_val_onehot_df, X_val_number_part], axis=1)
X_test_model = pd.concat([X_test_onehot_df, X_test_number_part], axis=1)

print("Final model data created.")
print("X_train_model:", X_train_model.shape)
print("X_val_model:", X_val_model.shape)
print("X_test_model:", X_test_model.shape)

print("\nMissing values after preprocessing:")
print("Train:", X_train_model.isna().sum().sum())
print("Validation:", X_val_model.isna().sum().sum())
print("Test:", X_test_model.isna().sum().sum())

Final model data created.
X_train_model: (3804, 104)
X_val_model: (952, 104)
X_test_model: (4749, 104)

Missing values after preprocessing:
Train: 0
Validation: 0
Test: 0


## **Helper functions for models**

In [60]:
def get_model_scores(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_data)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 0.000000001)
        return scores

    return model.predict(X_data)


def make_submission_file(test_scores, file_path):
    submission = example_submission.copy()
    submission[ID_COL] = test_ids.values
    submission[TARGET_COL] = test_scores
    submission = submission[example_submission.columns]

    assert list(submission.columns) == list(example_submission.columns)
    assert submission.shape[0] == example_submission.shape[0]
    assert submission[TARGET_COL].isna().sum() == 0

    submission.to_csv(file_path, index=False)

    return submission


def run_model(person_code, model_name, model):
    person_name = team_members[person_code]

    print("=" * 80)
    print("Person:", person_name)
    print("Model:", model_name)

    model.fit(X_train_model, y_train)

    val_scores = get_model_scores(model, X_val_model)
    val_predictions = (val_scores >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_val, val_scores)
    accuracy = accuracy_score(y_val, val_predictions)
    precision = precision_score(y_val, val_predictions, zero_division=0)
    recall = recall_score(y_val, val_predictions, zero_division=0)
    f1 = f1_score(y_val, val_predictions, zero_division=0)

    print("ROC-AUC:", round(roc_auc, 4))
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1:", round(f1, 4))

    print("\nClassification report:")
    print(classification_report(y_val, val_predictions))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, val_predictions))

    test_scores = get_model_scores(model, X_test_model)

    output_file = SUBMISSION_FOLDER / f"{person_code}_{person_name}_{model_name}.csv"
    submission = make_submission_file(test_scores, output_file)

    result = {
        "person_code": person_code,
        "person_name": person_name,
        "model": model_name,
        "roc_auc": roc_auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "submission_file": str(output_file)
    }

    return result, model, val_scores, submission


all_results = []
trained_models = {}
validation_score_dict = {}

## **Mahesh models**

In [26]:
person_code = "person01"

try:
    ridge_calibrated = CalibratedClassifierCV(
        estimator=RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        cv=3
    )
except TypeError:
    ridge_calibrated = CalibratedClassifierCV(
        base_estimator=RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        cv=3
    )

mahesh_models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),

    "ridge_classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ridge_calibrated)
    ])
}

mahesh_results = []

for model_name, model in mahesh_models.items():
    result, trained_model, val_scores, submission = run_model(person_code, model_name, model)
    mahesh_results.append(result)
    all_results.append(result)
    trained_models[model_name] = trained_model
    validation_score_dict[model_name] = val_scores

mahesh_results_df = pd.DataFrame(mahesh_results)
display(mahesh_results_df)
mahesh_results_df.to_csv(OUTPUT_FOLDER / "mahesh_results.csv", index=False)

Training: knn
ROC-AUC: 0.8205
Accuracy: 0.7857
Precision: 0.7635
Recall: 0.4984
F1: 0.6031

Classification report:
              precision    recall  f1-score   support

           0       0.79      0.93      0.85       641
           1       0.76      0.50      0.60       311

    accuracy                           0.79       952
   macro avg       0.78      0.71      0.73       952
weighted avg       0.78      0.79      0.77       952


Confusion matrix:
[[593  48]
 [156 155]]

Saved test prediction file: person02_knn.csv
Training: svc_rbf
ROC-AUC: 0.8519
Accuracy: 0.8099
Precision: 0.7462
Recall: 0.6334
F1: 0.6852

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.86       641
           1       0.75      0.63      0.69       311

    accuracy                           0.81       952
   macro avg       0.79      0.76      0.77       952
weighted avg       0.81      0.81      0.81       952


Confusion matrix:
[[574

,model,roc_auc,accuracy,precision,recall,f1
0,knn,0.820538,0.785714,0.763547,0.498392,0.603113
1,svc_rbf,0.851950,0.809874,0.746212,0.633441,0.685217


## **Waleed models**

In [61]:
person_code = "person02"

waleed_models = {
    "knn": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=25, weights="distance"))
    ]),

    "svc_rbf": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(C=1.0, kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE))
    ])
}

waleed_results = []

for model_name, model in waleed_models.items():
    result, trained_model, val_scores, submission = run_model(person_code, model_name, model)
    waleed_results.append(result)
    all_results.append(result)
    trained_models[model_name] = trained_model
    validation_score_dict[model_name] = val_scores

waleed_results_df = pd.DataFrame(waleed_results)
display(waleed_results_df)
waleed_results_df.to_csv(OUTPUT_FOLDER / "waleed_results.csv", index=False)

Person: Waleed
Model: knn
ROC-AUC: 0.8087
Accuracy: 0.7763
Precision: 0.75
Recall: 0.4727
F1: 0.5799

Classification report:
              precision    recall  f1-score   support

           0       0.78      0.92      0.85       641
           1       0.75      0.47      0.58       311

    accuracy                           0.78       952
   macro avg       0.77      0.70      0.71       952
weighted avg       0.77      0.78      0.76       952


Confusion matrix:
[[592  49]
 [164 147]]
Person: Waleed
Model: svc_rbf
ROC-AUC: 0.8476
Accuracy: 0.8057
Precision: 0.7386
Recall: 0.627
F1: 0.6783

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       641
           1       0.74      0.63      0.68       311

    accuracy                           0.81       952
   macro avg       0.79      0.76      0.77       952
weighted avg       0.80      0.81      0.80       952


Confusion matrix:
[[572  69]
 [116 195]]


,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file
0,person02,Waleed,knn,0.808679,0.776261,0.750000,0.472669,0.579882,outputs/submissions/person02_Waleed_knn.csv
1,person02,Waleed,svc_rbf,0.847646,0.805672,0.738636,0.627010,0.678261,outputs/submissions/person02_Waleed_svc_rbf.csv


## **Gopal models**

In [62]:
person_code = "person03"

gopal_models = {
    "gaussian_nb": GaussianNB(),

    "decision_tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

gopal_results = []

for model_name, model in gopal_models.items():
    result, trained_model, val_scores, submission = run_model(person_code, model_name, model)
    gopal_results.append(result)
    all_results.append(result)
    trained_models[model_name] = trained_model
    validation_score_dict[model_name] = val_scores

gopal_results_df = pd.DataFrame(gopal_results)
display(gopal_results_df)
gopal_results_df.to_csv(OUTPUT_FOLDER / "gopal_results.csv", index=False)

Person: Gopal
Model: gaussian_nb
ROC-AUC: 0.7562
Accuracy: 0.5777
Precision: 0.4253
Recall: 0.8328
F1: 0.563

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.45      0.59       641
           1       0.43      0.83      0.56       311

    accuracy                           0.58       952
   macro avg       0.64      0.64      0.58       952
weighted avg       0.71      0.58      0.58       952


Confusion matrix:
[[291 350]
 [ 52 259]]
Person: Gopal
Model: decision_tree
ROC-AUC: 0.8328
Accuracy: 0.7626
Precision: 0.6115
Recall: 0.7492
F1: 0.6734

Classification report:
              precision    recall  f1-score   support

           0       0.86      0.77      0.81       641
           1       0.61      0.75      0.67       311

    accuracy                           0.76       952
   macro avg       0.74      0.76      0.74       952
weighted avg       0.78      0.76      0.77       952


Confusion matrix:
[[493 148]
 [ 78

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file
0,person03,Gopal,gaussian_nb,0.756196,0.577731,0.425287,0.832797,0.563043,outputs/submissions/person03_Gopal_gaussian_nb...
1,person03,Gopal,decision_tree,0.832833,0.762605,0.611549,0.749196,0.673410,outputs/submissions/person03_Gopal_decision_tr...


## **Lavanya models**

In [63]:
person_code = "person04"

lavanya_models = {
    "random_forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "extra_trees": ExtraTreesClassifier(
        n_estimators=700,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

lavanya_results = []

for model_name, model in lavanya_models.items():
    result, trained_model, val_scores, submission = run_model(person_code, model_name, model)
    lavanya_results.append(result)
    all_results.append(result)
    trained_models[model_name] = trained_model
    validation_score_dict[model_name] = val_scores

lavanya_results_df = pd.DataFrame(lavanya_results)
display(lavanya_results_df)
lavanya_results_df.to_csv(OUTPUT_FOLDER / "lavanya_results.csv", index=False)

Person: Lavanya
Model: random_forest
ROC-AUC: 0.8565
Accuracy: 0.7941
Precision: 0.6898
Recall: 0.672
F1: 0.6808

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.85       641
           1       0.69      0.67      0.68       311

    accuracy                           0.79       952
   macro avg       0.77      0.76      0.76       952
weighted avg       0.79      0.79      0.79       952


Confusion matrix:
[[547  94]
 [102 209]]
Person: Lavanya
Model: extra_trees
ROC-AUC: 0.8521
Accuracy: 0.7952
Precision: 0.6933
Recall: 0.6688
F1: 0.6809

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       641
           1       0.69      0.67      0.68       311

    accuracy                           0.80       952
   macro avg       0.77      0.76      0.77       952
weighted avg       0.79      0.80      0.79       952


Confusion matrix:
[[549  92]
 

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file
0,person04,Lavanya,random_forest,0.856529,0.794118,0.689769,0.672026,0.680782,outputs/submissions/person04_Lavanya_random_fo...
1,person04,Lavanya,extra_trees,0.852100,0.795168,0.693333,0.668810,0.680851,outputs/submissions/person04_Lavanya_extra_tre...


## **Rakesh models**

In [64]:
person_code = "person05"

rakesh_models = {
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        random_state=RANDOM_STATE
    ),

    "adaboost": AdaBoostClassifier(
        n_estimators=300,
        learning_rate=0.03,
        random_state=RANDOM_STATE
    )
}

rakesh_results = []

for model_name, model in rakesh_models.items():
    result, trained_model, val_scores, submission = run_model(person_code, model_name, model)
    rakesh_results.append(result)
    all_results.append(result)
    trained_models[model_name] = trained_model
    validation_score_dict[model_name] = val_scores

rakesh_results_df = pd.DataFrame(rakesh_results)
display(rakesh_results_df)
rakesh_results_df.to_csv(OUTPUT_FOLDER / "rakesh_results.csv", index=False)

Person: Rakesh
Model: gradient_boosting
ROC-AUC: 0.8637
Accuracy: 0.8151
Precision: 0.7626
Recall: 0.6302
F1: 0.6901

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.87       641
           1       0.76      0.63      0.69       311

    accuracy                           0.82       952
   macro avg       0.80      0.77      0.78       952
weighted avg       0.81      0.82      0.81       952


Confusion matrix:
[[580  61]
 [115 196]]
Person: Rakesh
Model: adaboost
ROC-AUC: 0.8461
Accuracy: 0.7931
Precision: 0.7689
Recall: 0.5241
F1: 0.6233

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.92      0.86       641
           1       0.77      0.52      0.62       311

    accuracy                           0.79       952
   macro avg       0.78      0.72      0.74       952
weighted avg       0.79      0.79      0.78       952


Confusion matrix:
[[592  49]
 

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file
0,person05,Rakesh,gradient_boosting,0.863650,0.815126,0.762646,0.630225,0.690141,outputs/submissions/person05_Rakesh_gradient_b...
1,person05,Rakesh,adaboost,0.846073,0.793067,0.768868,0.524116,0.623327,outputs/submissions/person05_Rakesh_adaboost.csv


## **Thivas models**

In [65]:
person_code = "person06"

thivas_models = {
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.03,
        max_leaf_nodes=31,
        l2_regularization=0.1,
        random_state=RANDOM_STATE
    ),

    "mlp_neural_network": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=500,
            early_stopping=True,
            random_state=RANDOM_STATE
        ))
    ])
}

thivas_results = []

for model_name, model in thivas_models.items():
    result, trained_model, val_scores, submission = run_model(person_code, model_name, model)
    thivas_results.append(result)
    all_results.append(result)
    trained_models[model_name] = trained_model
    validation_score_dict[model_name] = val_scores

thivas_results_df = pd.DataFrame(thivas_results)
display(thivas_results_df)
thivas_results_df.to_csv(OUTPUT_FOLDER / "thivas_results.csv", index=False)

Person: Thivas
Model: hist_gradient_boosting
ROC-AUC: 0.8512
Accuracy: 0.7983
Precision: 0.7245
Recall: 0.6174
F1: 0.6667

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       641
           1       0.72      0.62      0.67       311

    accuracy                           0.80       952
   macro avg       0.78      0.75      0.76       952
weighted avg       0.79      0.80      0.79       952


Confusion matrix:
[[568  73]
 [119 192]]
Person: Thivas
Model: mlp_neural_network
ROC-AUC: 0.8472
Accuracy: 0.8088
Precision: 0.7295
Recall: 0.6592
F1: 0.6926

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       641
           1       0.73      0.66      0.69       311

    accuracy                           0.81       952
   macro avg       0.79      0.77      0.78       952
weighted avg       0.81      0.81      0.81       952


Confusion matri

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file
0,person06,Thivas,hist_gradient_boosting,0.851152,0.798319,0.724528,0.617363,0.666667,outputs/submissions/person06_Thivas_hist_gradi...
1,person06,Thivas,mlp_neural_network,0.847209,0.808824,0.729537,0.659164,0.692568,outputs/submissions/person06_Thivas_mlp_neural...


## **Combine all model results**

In [66]:
all_results_df = pd.DataFrame(all_results)
all_results_df = all_results_df.sort_values("f1", ascending=False)

display(all_results_df)

all_results_df.to_csv(OUTPUT_FOLDER / "all_model_results_default_threshold.csv", index=False)

fig = px.bar(
    all_results_df,
    x="model",
    y="f1",
    color="person_name",
    text="f1",
    title="Model F1 Scores at Default 0.5 Threshold"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    xaxis_title="Model",
    yaxis_title="F1 score",
    xaxis_tickangle=-45
)

fig.show()
fig.write_html(PLOT_FOLDER / "model_f1_default_threshold.html")

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file
9,person06,Thivas,mlp_neural_network,0.847209,0.808824,0.729537,0.659164,0.692568,outputs/submissions/person06_Thivas_mlp_neural...
6,person05,Rakesh,gradient_boosting,0.863650,0.815126,0.762646,0.630225,0.690141,outputs/submissions/person05_Rakesh_gradient_b...
5,person04,Lavanya,extra_trees,0.852100,0.795168,0.693333,0.668810,0.680851,outputs/submissions/person04_Lavanya_extra_tre...
4,person04,Lavanya,random_forest,0.856529,0.794118,0.689769,0.672026,0.680782,outputs/submissions/person04_Lavanya_random_fo...
1,person02,Waleed,svc_rbf,0.847646,0.805672,0.738636,0.627010,0.678261,outputs/submissions/person02_Waleed_svc_rbf.csv
3,person03,Gopal,decision_tree,0.832833,0.762605,0.611549,0.749196,0.673410,outputs/submissions/person03_Gopal_decision_tr...
8,person06,Thivas,hist_gradient_boosting,0.851152,0.798319,0.724528,0.617363,0.666667,outputs/submissions/person06_Thivas_hist_gradi...
7,person05,Rakesh,adaboost,0.846073,0.793067,0.768868,0.524116,0.623327,outputs/submissions/person05_Rakesh_adaboost.csv
0,person02,Waleed,knn,0.808679,0.776261,0.750000,0.472669,0.579882,outputs/submissions/person02_Waleed_knn.csv
2,person03,Gopal,gaussian_nb,0.756196,0.577731,0.425287,0.832797,0.563043,outputs/submissions/person03_Gopal_gaussian_nb...


## **Find best thresholds for each model**

In [67]:
def find_best_threshold(y_true, scores):
    best_threshold = 0.5
    best_f1 = -1

    for threshold in np.arange(0.05, 0.96, 0.01):
        predictions = (scores >= threshold).astype(int)
        this_f1 = f1_score(y_true, predictions, zero_division=0)

        if this_f1 > best_f1:
            best_f1 = this_f1
            best_threshold = threshold

    return best_threshold, best_f1


threshold_results = []

for model_name, val_scores in validation_score_dict.items():
    best_threshold, best_f1 = find_best_threshold(y_val, val_scores)

    best_predictions = (val_scores >= best_threshold).astype(int)

    threshold_results.append({
        "model": model_name,
        "best_threshold": best_threshold,
        "f1_tuned": f1_score(y_val, best_predictions, zero_division=0),
        "accuracy_tuned": accuracy_score(y_val, best_predictions),
        "precision_tuned": precision_score(y_val, best_predictions, zero_division=0),
        "recall_tuned": recall_score(y_val, best_predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_val, val_scores)
    })

threshold_results_df = pd.DataFrame(threshold_results)
threshold_results_df = threshold_results_df.sort_values("f1_tuned", ascending=False)

display(threshold_results_df)

threshold_results_df.to_csv(OUTPUT_FOLDER / "threshold_tuning_results.csv", index=False)

,model,best_threshold,f1_tuned,accuracy_tuned,precision_tuned,recall_tuned,roc_auc
6,gradient_boosting,0.41,0.716129,0.815126,0.718447,0.713826,0.863650
9,mlp_neural_network,0.44,0.707395,0.808824,0.707395,0.707395,0.847209
5,extra_trees,0.41,0.704678,0.787815,0.646113,0.774920,0.852100
4,random_forest,0.45,0.701649,0.790966,0.657303,0.752412,0.856529
8,hist_gradient_boosting,0.36,0.699690,0.796218,0.674627,0.726688,0.851152
1,svc_rbf,0.44,0.697900,0.803571,0.701299,0.694534,0.847646
7,adaboost,0.35,0.691983,0.769958,0.615000,0.790997,0.846073
3,decision_tree,0.50,0.673410,0.762605,0.611549,0.749196,0.832833
0,knn,0.36,0.655172,0.768908,0.639144,0.672026,0.808679
2,gaussian_nb,0.95,0.581986,0.619748,0.454054,0.810289,0.756196


## **Final results**

In [68]:
final_results_df = all_results_df.merge(
    threshold_results_df,
    on=["model", "roc_auc"],
    how="left"
)

final_results_df = final_results_df.sort_values("f1_tuned", ascending=False)

display(final_results_df)

final_results_df.to_csv(OUTPUT_FOLDER / "final_results.csv", index=False)

fig = px.bar(
    final_results_df,
    x="model",
    y="f1_tuned",
    color="person_name",
    text="f1_tuned",
    title="Final Model Comparison by Tuned F1 Score"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    xaxis_title="Model",
    yaxis_title="Tuned F1 Score",
    xaxis_tickangle=-45
)

fig.show()
fig.write_html(PLOT_FOLDER / "final_tuned_f1_scores.html")

fig = px.scatter(
    final_results_df,
    x="roc_auc",
    y="f1_tuned",
    color="person_name",
    text="model",
    title="ROC-AUC vs Tuned F1 Score"
)

fig.update_traces(textposition="top center")
fig.update_layout(
    xaxis_title="ROC-AUC",
    yaxis_title="Tuned F1 Score"
)

fig.show()
fig.write_html(PLOT_FOLDER / "roc_auc_vs_tuned_f1.html")

heatmap_df = final_results_df[
    ["model", "accuracy_tuned", "precision_tuned", "recall_tuned", "f1_tuned", "roc_auc"]
].set_index("model")

fig = px.imshow(
    heatmap_df,
    text_auto=".3f",
    aspect="auto",
    title="Final Model Metrics Heatmap"
)

fig.show()
fig.write_html(PLOT_FOLDER / "final_model_metrics_heatmap.html")

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file,best_threshold,f1_tuned,accuracy_tuned,precision_tuned,recall_tuned
1,person05,Rakesh,gradient_boosting,0.863650,0.815126,0.762646,0.630225,0.690141,outputs/submissions/person05_Rakesh_gradient_b...,0.41,0.716129,0.815126,0.718447,0.713826
0,person06,Thivas,mlp_neural_network,0.847209,0.808824,0.729537,0.659164,0.692568,outputs/submissions/person06_Thivas_mlp_neural...,0.44,0.707395,0.808824,0.707395,0.707395
2,person04,Lavanya,extra_trees,0.852100,0.795168,0.693333,0.668810,0.680851,outputs/submissions/person04_Lavanya_extra_tre...,0.41,0.704678,0.787815,0.646113,0.774920
3,person04,Lavanya,random_forest,0.856529,0.794118,0.689769,0.672026,0.680782,outputs/submissions/person04_Lavanya_random_fo...,0.45,0.701649,0.790966,0.657303,0.752412
6,person06,Thivas,hist_gradient_boosting,0.851152,0.798319,0.724528,0.617363,0.666667,outputs/submissions/person06_Thivas_hist_gradi...,0.36,0.699690,0.796218,0.674627,0.726688
4,person02,Waleed,svc_rbf,0.847646,0.805672,0.738636,0.627010,0.678261,outputs/submissions/person02_Waleed_svc_rbf.csv,0.44,0.697900,0.803571,0.701299,0.694534
7,person05,Rakesh,adaboost,0.846073,0.793067,0.768868,0.524116,0.623327,outputs/submissions/person05_Rakesh_adaboost.csv,0.35,0.691983,0.769958,0.615000,0.790997
5,person03,Gopal,decision_tree,0.832833,0.762605,0.611549,0.749196,0.673410,outputs/submissions/person03_Gopal_decision_tr...,0.50,0.673410,0.762605,0.611549,0.749196
8,person02,Waleed,knn,0.808679,0.776261,0.750000,0.472669,0.579882,outputs/submissions/person02_Waleed_knn.csv,0.36,0.655172,0.768908,0.639144,0.672026
9,person03,Gopal,gaussian_nb,0.756196,0.577731,0.425287,0.832797,0.563043,outputs/submissions/person03_Gopal_gaussian_nb...,0.95,0.581986,0.619748,0.454054,0.810289


## **Picking 5 models**

In [69]:
top_5_models = final_results_df.head(5).copy()

display(top_5_models)

top_5_models.to_csv(OUTPUT_FOLDER / "top_5_models_by_tuned_f1.csv", index=False)

print("Top 5 models based on tuned F1:")

for rank, row in enumerate(top_5_models.itertuples(index=False), start=1):
    print(
        rank,
        row.model,
        "| Person:",
        row.person_name,
        "| Tuned F1:",
        round(row.f1_tuned, 4),
        "| Best threshold:",
        round(row.best_threshold, 2)
    )

fig = px.bar(
    top_5_models,
    x="model",
    y="f1_tuned",
    color="person_name",
    text="f1_tuned",
    title="Top 5 Models by Tuned F1 Score"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    xaxis_title="Model",
    yaxis_title="Tuned F1 Score",
    xaxis_tickangle=-45
)

fig.show()
fig.write_html(PLOT_FOLDER / "top_5_models_by_tuned_f1.html")

,person_code,person_name,model,roc_auc,accuracy,precision,recall,f1,submission_file,best_threshold,f1_tuned,accuracy_tuned,precision_tuned,recall_tuned
1,person05,Rakesh,gradient_boosting,0.863650,0.815126,0.762646,0.630225,0.690141,outputs/submissions/person05_Rakesh_gradient_b...,0.41,0.716129,0.815126,0.718447,0.713826
0,person06,Thivas,mlp_neural_network,0.847209,0.808824,0.729537,0.659164,0.692568,outputs/submissions/person06_Thivas_mlp_neural...,0.44,0.707395,0.808824,0.707395,0.707395
2,person04,Lavanya,extra_trees,0.852100,0.795168,0.693333,0.668810,0.680851,outputs/submissions/person04_Lavanya_extra_tre...,0.41,0.704678,0.787815,0.646113,0.774920
3,person04,Lavanya,random_forest,0.856529,0.794118,0.689769,0.672026,0.680782,outputs/submissions/person04_Lavanya_random_fo...,0.45,0.701649,0.790966,0.657303,0.752412
6,person06,Thivas,hist_gradient_boosting,0.851152,0.798319,0.724528,0.617363,0.666667,outputs/submissions/person06_Thivas_hist_gradi...,0.36,0.699690,0.796218,0.674627,0.726688


Top 5 models based on tuned F1:
1 gradient_boosting | Person: Rakesh | Tuned F1: 0.7161 | Best threshold: 0.41
2 mlp_neural_network | Person: Thivas | Tuned F1: 0.7074 | Best threshold: 0.44
3 extra_trees | Person: Lavanya | Tuned F1: 0.7047 | Best threshold: 0.41
4 random_forest | Person: Lavanya | Tuned F1: 0.7016 | Best threshold: 0.45
5 hist_gradient_boosting | Person: Thivas | Tuned F1: 0.6997 | Best threshold: 0.36


## **Top 5 submission files**

In [70]:
ranked_files = []

for rank, row in enumerate(top_5_models.itertuples(index=False), start=1):
    model_name = row.model
    person_name = row.person_name

    print("=" * 80)
    print("Rank:", rank)
    print("Model:", model_name)
    print("Person:", person_name)
    print("Validation tuned F1:", round(row.f1_tuned, 4))

    model = trained_models[model_name]
    model.fit(X_train_model, y_train)

    test_scores = get_model_scores(model, X_test_model)

    file_name = f"rank{rank:02d}_f1_{person_name}_{model_name}.csv"
    file_path = SUBMISSION_FOLDER / file_name

    submission = make_submission_file(test_scores, file_path)

    ranked_files.append(file_path)

    print("Saved:", file_path)
    display(submission.head())

print("\nTop 5 submission files:")
for file in ranked_files:
    print(file)

Rank: 1
Model: gradient_boosting
Person: Rakesh
Validation tuned F1: 0.7161
Saved: outputs/submissions/rank01_f1_Rakesh_gradient_boosting.csv


,respondent_id,covid_vaccine
0,4757,0.177032
1,4758,0.134874
2,4759,0.189762
3,4760,0.587589
4,4761,0.079510


Rank: 2
Model: mlp_neural_network
Person: Thivas
Validation tuned F1: 0.7074
Saved: outputs/submissions/rank02_f1_Thivas_mlp_neural_network.csv


,respondent_id,covid_vaccine
0,4757,0.257616
1,4758,0.180407
2,4759,0.290876
3,4760,0.456469
4,4761,0.075390


Rank: 3
Model: extra_trees
Person: Lavanya
Validation tuned F1: 0.7047
Saved: outputs/submissions/rank03_f1_Lavanya_extra_trees.csv


...

Rank: 4
Model: random_forest
Person: Lavanya
Validation tuned F1: 0.7016
Saved: outputs/submissions/rank04_f1_Lavanya_random_forest.csv


,respondent_id,covid_vaccine
0,4757,0.385222
1,4758,0.204441
2,4759,0.357231
3,4760,0.468817
4,4761,0.160125


Rank: 5
Model: hist_gradient_boosting
Person: Thivas
Validation tuned F1: 0.6997
Saved: outputs/submissions/rank05_f1_Thivas_hist_gradient_boosting.csv


,respondent_id,covid_vaccine
0,4757,0.187240
1,4758,0.105765
2,4759,0.234577
3,4760,0.477416
4,4761,0.051939



Top 5 submission files:
outputs/submissions/rank01_f1_Rakesh_gradient_boosting.csv
outputs/submissions/rank02_f1_Thivas_mlp_neural_network.csv
outputs/submissions/rank03_f1_Lavanya_extra_trees.csv
outputs/submissions/rank04_f1_Lavanya_random_forest.csv
outputs/submissions/rank05_f1_Thivas_hist_gradient_boosting.csv
